# Arctic snow and ice, 1980–2015

Monthly timeseries of the CICE fields `aice`, `hi`, `hs` and `snowfrac` for the
POP+CAMulator coupled checkpoints `ck33`, `ck36`, `ck64`, `ck66`, `ck69`, `ck70`,
built the same way as the sea-ice area timeseries in `../nino34_drift.ipynb`.

Both hemispheres are computed and saved — the plot draws the **Arctic**; flip `HEMI`
in the last cell for the Antarctic.

Everything is read in place. Nothing is written outside `/glade/work/tvoeller`.

### baseline check

Does the CREDIT baseline carry `aice` / `hi` / `hs`? If not, the baseline line is
left as a placeholder and only the checkpoint climatologies are drawn.

In [ ]:
import xarray as xr

BASELINE_ZARR = (
    "/glade/derecho/scratch/wchapman/b_credit_runs/"
    "b.e21.CREDIT_climate_branch_1980_1980_zmdata_ERA5scaled_zmdata_Qtot.zarr"
)

# The three the baseline is being checked for, plus snowfrac which goes into the
# checkpoint dataset for later regardless.
CHECK_VARS = ["aice", "hi", "hs"]
ICE_VARS = ["aice", "hi", "hs", "snowfrac"]

ds_base = xr.open_zarr(BASELINE_ZARR, chunks={})

present = [v for v in CHECK_VARS if v in ds_base.variables]
missing = [v for v in CHECK_VARS if v not in ds_base.variables]

print(f"baseline store: {BASELINE_ZARR.split('/')[-1]}")
print(f"  present: {present or 'none'}")
print(f"  missing: {missing or 'none'}")
print(f"  {len(ds_base.data_vars)} data variables, e.g. {sorted(ds_base.data_vars)[:8]}")

# ------------------------------------------------------------------
# PLACEHOLDER
# ------------------------------------------------------------------
# This store is the CAM *atmosphere* history: it has no sea-ice state, so none of
# aice / hi / hs are in it. The closest thing it does carry is ICEFRAC, the
# atmosphere's sea-ice fraction, which ice_ens/data.py already uses as the
# reference for aice-like comparisons -- but there is no equivalent at all for the
# ice or snow *thickness*.
#
# So: no baseline line is drawn. When the right file arrives, point
# BASELINE_FILE at it and give BASELINE_VARS the name each of ICE_VARS goes by in
# it; the plot cell picks it up with no other change.
BASELINE_FILE = None          # e.g. "/glade/.../cice_baseline_climatology.nc"
BASELINE_VARS = {}            # e.g. {"aice": "aice", "hi": "hi", "hs": "hs"}

BASELINE_HAS_ICE = not missing
if not BASELINE_HAS_ICE:
    print(f"\n-> no baseline sea ice in this store; leaving a placeholder for {missing}")

### data process

One pass per checkpoint over its CICE monthly history. Only the four ice fields and
the two grid fields are read off disk.

Each field is reduced to a hemispheric integral against the T-cell area, which is
one operation for all four — `aice` and `snowfrac` are fractions, so the integral is
an **area**; `hi` and `hs` are grid-cell *mean* thicknesses, so the integral is
already a concentration-weighted **volume**.

In [ ]:
import os

import numpy as np
import pandas as pd
import xarray as xr

from ice_ens import data

CHECKPOINTS = ["ck33", "ck36", "ck64", "ck66", "ck69", "ck70"]

# Full desired date range (kept for later reindexing). Stops at the end of 2015
# because that is where the training data stops -- ck70 runs on past it, and
# everything after is deliberately cut.
full_dates = pd.date_range("1980-01", "2015-12", freq="MS")

# What each integral is called and what it comes out in. "fraction" -> area,
# "thickness" -> volume.
INTEGRALS = {
    "aice":     ("ice_area",   "fraction",  "1e6 km^2",   "Total sea ice area"),
    "hi":       ("ice_volume", "thickness", "1e3 km^3",   "Total sea ice volume"),
    "hs":       ("snow_volume", "thickness", "1e3 km^3",  "Total snow volume on sea ice"),
    "snowfrac": ("snow_area",  "fraction",  "1e6 km^2",   "Snow-covered sea ice area"),
}

HEMISPHERES = {"arctic": lambda lat: lat > 66.5, "antarctic": lambda lat: lat < -66.5}


def cice_files(case):
    """The monthly CICE history files for one case, with the month each one covers.

    The date comes from the filename, not the time coordinate: CICE stamps the
    time axis at the *end* of the averaging interval, so trusting it shifts the
    whole seasonal cycle by a month (same reason ice_ens/data.py reads POP dates
    off the filename).
    """
    run_dir = os.path.join(data.CASE_ROOT, data.case_name(case), "run")
    stem = data.case_name(case)
    found = [
        (d, os.path.join(run_dir, f"{stem}.cice.h.{d.strftime('%Y-%m')}.nc"))
        for d in full_dates
    ]
    return [(d, f) for d, f in found if os.path.exists(f)]


def integrate(case):
    """Hemispheric integrals of ICE_VARS for one checkpoint, on the full monthly grid."""
    existing = cice_files(case)
    if not existing:
        raise FileNotFoundError(f"no CICE monthly files for {data.case_name(case)}")

    dates_have = pd.DatetimeIndex([d for d, _ in existing])
    files = [f for _, f in existing]
    print(
        f"{case}: found {len(files)} / {len(full_dates)} months "
        f"({dates_have[0]:%Y-%m} to {dates_have[-1]:%Y-%m}); "
        f"{len(full_dates) - len(files)} filled with NaN"
    )

    ds = xr.open_mfdataset(
        files,
        preprocess=lambda d: d[ICE_VARS + ["tarea", "TLAT"]],
        combine="nested",
        concat_dim="time",
        decode_times=False,
        parallel=True,
    )
    ds["time"] = dates_have

    tarea = ds["tarea"]
    tlat = ds["TLAT"]
    if "time" in tarea.dims:
        tarea = tarea.isel(time=0)   # static
    if "time" in tlat.dims:
        tlat = tlat.isel(time=0)     # static

    out = xr.Dataset()
    for var, (stem, kind, units, long_name) in INTEGRALS.items():
        # fraction * m^2 -> km^2 -> 1e6 km^2 ; m * m^2 -> m^3 -> 1e3 km^3
        weighted = ds[var] * tarea
        scale = 1.0e6 * 1.0e6 if kind == "fraction" else 1.0e12

        spatial_dims = [d for d in weighted.dims if d != "time"]
        for hemi, test in HEMISPHERES.items():
            series = weighted.where(test(tlat)).sum(dim=spatial_dims, skipna=True) / scale
            name = f"{hemi}_{stem}"
            out[name] = series
            out[name].attrs = {
                "units": units,
                "long_name": f"{long_name} ({hemi})",
                "source_variable": var,
                "missing_note": "NaN where the model had not yet produced that month",
            }

    out = out.compute()
    ds.close()

    # Reindex onto the full 1980-01 .. 2014-12 grid, NaN where nothing was written.
    out = out.reindex(time=full_dates, fill_value=np.nan)
    out["time"].attrs = {"long_name": "time"}
    out.attrs["case"] = data.case_name(case)
    out.attrs["label"] = case
    out.attrs["months_available"] = len(files)
    return out


# ------------------------------------------------------------------
# Run every checkpoint and save one file each
# ------------------------------------------------------------------
series = {}
for case in CHECKPOINTS:
    out = integrate(case)
    out_file = os.path.join(
        data.out_dir(case, "snow"), f"snow_ice_integrals_1980_2015_{case}.nc"
    )
    out.to_netcdf(out_file)
    series[case] = out
    print(f"    saved {out_file}")

coverage = pd.DataFrame(
    {
        case: {
            "months": int(ds.attrs["months_available"]),
            "first": str(ds["time"].values[0])[:7],
            "last": str(
                ds["time"].where(ds["arctic_ice_area"].notnull(), drop=True).values[-1]
            )[:7],
        }
        for case, ds in series.items()
    }
).T
coverage

### plot

Arctic monthly time series, 1980–2015, one panel per field.

The axis is the full 1980-01 … 2015-12 grid for every checkpoint, so a line simply
stops where that checkpoint ran out — the months after it are NaN, not zero, and leave
a gap rather than a drop to the floor. Each checkpoint's last month is in its legend
label, since they end at very different dates.

2015 is the cutoff because that is where the training data stops; ck70 goes on to
2025-04 and is cut here on purpose.

In [ ]:
import os

import matplotlib.dates as mdates
import matplotlib.pyplot as plt
import pandas as pd
import xarray as xr

from ice_ens import data

HEMI = "arctic"
PLOT_START, PLOT_END = pd.Timestamp("1980-01-01"), pd.Timestamp("2015-12-31")

colors = {
    "ck33": "tab:blue",
    "ck36": "tab:orange",
    "ck64": "tab:green",
    "ck66": "tab:red",
    "ck69": "tab:purple",
    "ck70": "tab:brown",
}

# ------------------------------------------------------------------
# 1. Where each checkpoint actually runs out
# ------------------------------------------------------------------
def last_month(ds):
    """The last month this checkpoint wrote, as a Timestamp."""
    valid = ds["time"].where(ds[f"{HEMI}_ice_area"].notnull(), drop=True)
    return pd.Timestamp(str(valid.values[-1]))


ends = {case: last_month(ds) for case, ds in series.items()}
for case, end in sorted(ends.items(), key=lambda kv: kv[1]):
    print(f"{case}: through {end:%Y-%m}")

# ------------------------------------------------------------------
# 2. Optional baseline (placeholder until the right file exists)
# ------------------------------------------------------------------
ds_base_ice = None
if BASELINE_FILE and os.path.exists(BASELINE_FILE):
    ds_base_ice = xr.open_dataset(BASELINE_FILE)
    print(f"loaded baseline: {BASELINE_FILE}")
else:
    print("no baseline file set -- drawing checkpoints only")

# ------------------------------------------------------------------
# 3. Draw: one panel per field, monthly, 1980-2015
# ------------------------------------------------------------------
panels = [(stem, units, long_name) for stem, _, units, long_name in INTEGRALS.values()]

fig, axes = plt.subplots(len(panels), 1, figsize=(13, 2.9 * len(panels)), sharex=True)

for ax, (stem, units, long_name) in zip(axes, panels):
    name = f"{HEMI}_{stem}"

    for case, ds in series.items():
        # Already reindexed onto the full monthly grid, so the months after this
        # checkpoint stops are NaN and matplotlib leaves them blank.
        ax.plot(ds["time"], ds[name], color=colors[case], lw=1.0,
                label=f"{case} (through {ends[case]:%Y-%m})")

    if ds_base_ice is not None and name in ds_base_ice:
        ax.plot(ds_base_ice["time"], ds_base_ice[name], color="black", lw=1.5,
                ls="--", label="CREDIT (baseline)")

    ax.set_title(long_name.split("(")[0].strip(), fontsize=10)
    ax.set_ylabel(f"({units})")
    ax.grid(True, alpha=0.3)
    ax.set_xlim(PLOT_START, PLOT_END)

axes[-1].set_xlabel("Year")
axes[-1].xaxis.set_major_locator(mdates.YearLocator(5))
axes[-1].xaxis.set_minor_locator(mdates.YearLocator(1))
axes[-1].xaxis.set_major_formatter(mdates.DateFormatter("%Y"))

fig.suptitle(
    f"{HEMI.capitalize()} sea ice and snow, 1980-2015 "
    "(blank where a checkpoint has no more data)",
    y=1.045,
)
# Legend above the panels rather than inside one -- in the top panel it sat right
# on the summer minima.
handles, labels = axes[0].get_legend_handles_labels()
fig.legend(handles, labels, loc="upper center", bbox_to_anchor=(0.5, 1.012),
           ncol=len(labels), fontsize=8, frameon=False)
fig.tight_layout()

fig_dir = os.path.join(data.OUTPUT_DIR, "snow")
os.makedirs(fig_dir, exist_ok=True)
fig_path = os.path.join(fig_dir, f"{HEMI}_snow_ice_timeseries_1980_2015.png")
fig.savefig(fig_path, dpi=150, bbox_inches="tight")
print(f"saved {fig_path}")
plt.show()